In [0]:
from config import load_config

cfg = load_config("config_databricks.yaml")

In [0]:
from pyspark.sql import functions as F
from config import load_config

cfg = load_config("config_databricks.yaml")

workflow = spark.table(cfg["fct_workflow_events_table"])

applications = (
    spark.table(cfg["bronze_applications_table"])
    .select(
        F.trim(F.col("application_id")).alias("application_id"),
        F.trim(F.col("job_id")).alias("job_id"),
        F.trim(F.col("candidate_id")).alias("candidate_id"),
        F.trim(F.col("apply_date")).alias("apply_date")
    )
    .dropDuplicates(["application_id"])
)

hired_data = (
    workflow
    .filter(F.col("new_status") == "Hired")
    .groupBy("application_id")
    .agg(F.min("event_timestamp").alias("hired_date"))
)

# Get latest timestamp per application
latest_event = (
    workflow
    .groupBy("application_id")
    .agg(F.max("event_timestamp").alias("latest_timestamp"))
)

# Join back to get corresponding status
current_status_data = (
    latest_event.alias("l")
    .join(
        workflow.alias("w"),
        (F.col("l.application_id") == F.col("w.application_id")) &
        (F.col("l.latest_timestamp") == F.col("w.event_timestamp")),
        "left"
    )
    .select(
        F.col("l.application_id").alias("application_id"),
        F.col("w.new_status").alias("current_status")
    )
)

#is_hired flag and join
fct_applications = (
    applications
    .join(hired_data, on="application_id", how="left")
    .join(current_status_data, on="application_id", how="left")
    .withColumn("is_hired", F.when(F.col("hired_date").isNotNull(), 1).otherwise(0))
)

#Write table
fct_applications.write.format("delta").mode("overwrite").saveAsTable(cfg["fct_applications_table"])

In [0]:
spark.sql('SELECT COUNT(*) FROM fct_applications;').show()

spark.sql('SELECT application_id, COUNT(*) FROM fct_applications GROUP BY application_id HAVING COUNT(*) > 1;').show()

spark.sql('SELECT * FROM fct_applications WHERE is_hired = 1 LIMIT 10;').show()